# DA016-parent A/B/C ablation analysis

## TL;DR

B gives the most robust validation improvement over A, while C has the best single-run test mAP50-95. C's test gain is concentrated in truck and freight_car, and its residual hurts validation relative to B, so C is promising but not yet robustly established. The learned Prompts contain per-object signal (balanced accuracy about 0.67–0.69 and correlation about 0.42–0.45), but are only moderately accurate and slightly over-predict RGB reliability.

## Context & Methods

A is the unchanged DA016 parent; B adds dual reliability Prompt supervision without feeding Prompt into the fusion gate; C adds the same supervision plus a zero-initialized bounded Prompt residual. The analysis uses each completed 100-epoch run, its `best.pt` test output on the M2D-LIF test labels, and validation curves from `results.csv`. Empty failed placeholder directories are excluded. Test speed is not treated as a controlled comparison because A and B/C were evaluated on different GPUs.

In [1]:
from pathlib import Path
import csv, re, statistics
from pprint import pprint

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'runs/DroneVehicle_OBB_FusionTransfer').is_dir())
BASE = ROOT / 'runs/DroneVehicle_OBB_FusionTransfer'
RUNS = {
    'A': BASE / 'DA016-A_OriginalSemanticDisagreementLAF-P34_M2DLIFLabels_v1',
    'B': BASE / 'DA016-B_DualReliabilityPromptAuxOnly-P34_M2DLIFLabels_v12',
    'C': BASE / 'DA016-C_DualReliabilityPrompt-ZeroInitBoundedResidual-P34_M2DLIFLabels_v12',
}
for name, path in RUNS.items():
    assert (path / 'results.csv').is_file(), (name, path)
    assert (path / 'test_m2dlif/test.txt').is_file(), (name, path)
    assert (path / 'weights/best.pt').is_file(), (name, path)

## Data

The following parsers retain exact CSV precision for validation metrics and parse the rounded values printed by the test evaluator.

In [2]:
def read_results(path):
    with path.open(newline='') as f:
        rows = []
        for row in csv.DictReader(f):
            rows.append({k.strip(): float(v) for k, v in row.items()})
    return rows

def read_test(path):
    text = path.read_text()
    meta = {}
    for key in ['labels', 'rgb', 'ir', 'class_map', 'split', 'imgsz', 'batch', 'device']:
        m = re.search(rf'^{key}: (.+)$', text, re.M)
        meta[key] = m.group(1) if m else None
    classes = {}
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 7 and parts[0] in {'all', 'car', 'truck', 'bus', 'van', 'freight_car'}:
            classes[parts[0]] = {
                'images': int(parts[1]), 'instances': int(parts[2]),
                'precision': float(parts[3]), 'recall': float(parts[4]),
                'mAP50': float(parts[5]), 'mAP50-95': float(parts[6]),
            }
    speed = re.search(r'Speed: ([0-9.]+)ms preprocess, ([0-9.]+)ms inference.*?([0-9.]+)ms postprocess', text)
    fps = re.search(r'forward FPS: ([0-9.]+)', text)
    meta['preprocess_ms'], meta['inference_ms'], meta['postprocess_ms'] = map(float, speed.groups())
    meta['fps'] = float(fps.group(1))
    return meta, classes

curves = {name: read_results(path / 'results.csv') for name, path in RUNS.items()}
tests = {name: read_test(path / 'test_m2dlif/test.txt') for name, path in RUNS.items()}
source_audit = [{
    'run': name, 'epochs': len(curves[name]), 'test_images': tests[name][1]['all']['images'],
    'test_instances': tests[name][1]['all']['instances'], 'labels': tests[name][0]['labels'],
    'test_device': tests[name][0]['device']
} for name in RUNS]
pprint(source_audit)

[{'epochs': 100,
  'labels': '/media/biiteam/新加卷1/biiteam/MCONG/datasets/M2D-LIFlabels/DroneVehicle_test_labels/labels/val',
  'run': 'A',
  'test_device': '1',
  'test_images': 8980,
  'test_instances': 159618},
 {'epochs': 100,
  'labels': '/media/biiteam/新加卷1/biiteam/MCONG/datasets/M2D-LIFlabels/DroneVehicle_test_labels/labels/val',
  'run': 'B',
  'test_device': '4',
  'test_images': 8980,
  'test_instances': 159618},
 {'epochs': 100,
  'labels': '/media/biiteam/新加卷1/biiteam/MCONG/datasets/M2D-LIFlabels/DroneVehicle_test_labels/labels/val',
  'run': 'C',
  'test_device': '4',
  'test_images': 8980,
  'test_instances': 159618}]


## Results

In [3]:
test_rows = {}
for name in RUNS:
    all_row = tests[name][1]['all']
    test_rows[name] = {**all_row, 'fps': tests[name][0]['fps'], 'device': tests[name][0]['device']}
pprint(test_rows)

test_delta = {name: {m: test_rows[name][m] - test_rows['A'][m] for m in ['precision', 'recall', 'mAP50', 'mAP50-95']} for name in ['B', 'C']}
pprint(test_delta)

class_rows = []
for cls in ['car', 'truck', 'bus', 'van', 'freight_car']:
    for name in RUNS:
        row = tests[name][1][cls]
        class_rows.append({'class': cls, 'run': name, 'instances': row['instances'], 'mAP50': row['mAP50'], 'mAP50-95': row['mAP50-95']})
class_pivot = {}
for cls in ['car', 'truck', 'bus', 'van', 'freight_car']:
    vals = {name: tests[name][1][cls]['mAP50-95'] for name in RUNS}
    class_pivot[cls] = {**vals, 'B-A': vals['B']-vals['A'], 'C-A': vals['C']-vals['A'], 'C-B': vals['C']-vals['B']}
pprint(class_pivot)

{'A': {'device': '1',
       'fps': 242.53,
       'images': 8980,
       'instances': 159618,
       'mAP50': 0.819,
       'mAP50-95': 0.667,
       'precision': 0.799,
       'recall': 0.789},
 'B': {'device': '4',
       'fps': 151.29,
       'images': 8980,
       'instances': 159618,
       'mAP50': 0.818,
       'mAP50-95': 0.668,
       'precision': 0.794,
       'recall': 0.793},
 'C': {'device': '4',
       'fps': 148.67,
       'images': 8980,
       'instances': 159618,
       'mAP50': 0.822,
       'mAP50-95': 0.67,
       'precision': 0.803,
       'recall': 0.79}}
{'B': {'mAP50': -0.0010000000000000009,
       'mAP50-95': 0.0010000000000000009,
       'precision': -0.0050000000000000044,
       'recall': 0.0040000000000000036},
 'C': {'mAP50': 0.0030000000000000027,
       'mAP50-95': 0.0030000000000000027,
       'precision': 0.0040000000000000036,
       'recall': 0.0010000000000000009}}
{'bus': {'A': 0.79,
         'B': 0.793,
         'B-A': 0.0030000000000000027,
  

In [4]:
metric = 'metrics/mAP50-95(B)'
val_rows = []
for name, rows in curves.items():
    best = max(rows, key=lambda r: r[metric])
    item = {'run': name, 'best_epoch': int(best['epoch']), 'best_val_mAP50-95': best[metric], 'last_val_mAP50-95': rows[-1][metric]}
    for n in (10, 20, 30):
        vals = [r[metric] for r in rows[-n:]]
        item[f'last{n}_mean'] = statistics.mean(vals)
        item[f'last{n}_std'] = statistics.pstdev(vals)
    val_rows.append(item)
pprint(val_rows)

for challenger in ['B', 'C']:
    diffs = [r1[metric] - r0[metric] for r1, r0 in zip(curves[challenger][-30:], curves['A'][-30:])]
    print(challenger, 'minus A, last-30 mean:', statistics.mean(diffs), 'positive epochs:', sum(d > 0 for d in diffs), '/30')
diffs_cb = [rc[metric] - rb[metric] for rc, rb in zip(curves['C'][-30:], curves['B'][-30:])]
print('C minus B, last-30 mean:', statistics.mean(diffs_cb), 'positive epochs:', sum(d > 0 for d in diffs_cb), '/30')

[{'best_epoch': 66,
  'best_val_mAP50-95': 0.71995,
  'last10_mean': 0.717469,
  'last10_std': 0.0001717236151494667,
  'last20_mean': 0.7178215,
  'last20_std': 0.00041756765918831166,
  'last30_mean': 0.7182616666666667,
  'last30_std': 0.0007319748781359983,
  'last_val_mAP50-95': 0.71743,
  'run': 'A'},
 {'best_epoch': 80,
  'best_val_mAP50-95': 0.72366,
  'last10_mean': 0.72194,
  'last10_std': 0.00033861482542855216,
  'last20_mean': 0.7225185,
  'last20_std': 0.0006435470068301263,
  'last30_mean': 0.7227036666666666,
  'last30_std': 0.0006187917438219676,
  'last_val_mAP50-95': 0.72164,
  'run': 'B'},
 {'best_epoch': 71,
  'best_val_mAP50-95': 0.72181,
  'last10_mean': 0.719423,
  'last10_std': 0.0004763202704063834,
  'last20_mean': 0.719896,
  'last20_std': 0.000595310003275604,
  'last30_mean': 0.7202223333333333,
  'last30_std': 0.0007115367562927177,
  'last_val_mAP50-95': 0.71888,
  'run': 'C'}]
B minus A, last-30 mean: 0.004442000000000005 positive epochs: 30 /30
C minus

In [5]:
prompt_rows = []
for name in ['B', 'C']:
    best = max(curves[name], key=lambda r: r[metric])
    for level in ['P3', 'P4']:
        rgb_recall = best[f'p2/{level}_rgb_win_recall']
        ir_recall = best[f'p2/{level}_ir_win_recall']
        prompt_rows.append({
            'run': name, 'level': level, 'epoch': int(best['epoch']),
            'hard_accuracy': best[f'p2/{level}_object_prompt_accuracy'],
            'balanced_accuracy': (rgb_recall + ir_recall) / 2,
            'correlation': best[f'p2/{level}_object_prompt_correlation'],
            'predicted_rgb': best[f'p2/{level}_object_prompt_rgb'],
            'target_rgb': best[f'p2/{level}_object_target_rgb'],
            'rgb_bias': best[f'p2/{level}_object_prompt_rgb'] - best[f'p2/{level}_object_target_rgb'],
        })
pprint(prompt_rows)

c_gain = [{
    'epoch': int(r['epoch']), 'P3_gain': r['p2/P3_prompt_residual_gain'], 'P4_gain': r['p2/P4_prompt_residual_gain']
} for r in curves['C']]
pprint([r for r in c_gain if r['epoch'] in {1, 5, 10, 20, 40, 60, 71, 100}])
print('C last-10 mean gains:', {k: statistics.mean(r[k] for r in c_gain[-10:]) for k in ['P3_gain', 'P4_gain']})

[{'balanced_accuracy': 0.672805,
  'correlation': 0.41954,
  'epoch': 80,
  'hard_accuracy': 0.68324,
  'level': 'P3',
  'predicted_rgb': 0.44429,
  'rgb_bias': 0.03290000000000004,
  'run': 'B',
  'target_rgb': 0.41139},
 {'balanced_accuracy': 0.68818,
  'correlation': 0.45426,
  'epoch': 80,
  'hard_accuracy': 0.68686,
  'level': 'P4',
  'predicted_rgb': 0.4472,
  'rgb_bias': 0.03581000000000001,
  'run': 'B',
  'target_rgb': 0.41139},
 {'balanced_accuracy': 0.671495,
  'correlation': 0.42237,
  'epoch': 71,
  'hard_accuracy': 0.68143,
  'level': 'P3',
  'predicted_rgb': 0.44539,
  'rgb_bias': 0.03176000000000001,
  'run': 'C',
  'target_rgb': 0.41363},
 {'balanced_accuracy': 0.68516,
  'correlation': 0.45406,
  'epoch': 71,
  'hard_accuracy': 0.68586,
  'level': 'P4',
  'predicted_rgb': 0.44656,
  'rgb_bias': 0.032930000000000015,
  'run': 'C',
  'target_rgb': 0.41363}]
[{'P3_gain': 3.5761e-07, 'P4_gain': -2.6239e-07, 'epoch': 1},
 {'P3_gain': -0.0011465, 'P4_gain': -0.00032735, 'ep

## Takeaways

1. **B is the strongest robustness result.** Its best validation mAP50-95 is about 0.0037 above A and its final-30-epoch mean is about 0.0044 above A, with B ahead at every one of those epochs. Its rounded test gain is only 0.001, so the auxiliary target is useful but the transfer is modest.
2. **C is the best single test run, not yet the most reliable conclusion.** C improves test mAP50-95 by 0.003 over A, but remains about 0.0025 below B throughout the last 30 validation epochs. Its test gain is concentrated in truck and freight_car rather than all classes.
3. **Prompt is not merely a copied global IR prior.** Despite an IR-majority target, balanced accuracy is roughly 0.67–0.69 and correlation is 0.42–0.45, showing meaningful per-object variation. It is still only moderately predictive and has a small RGB-overprediction bias.
4. **The useful residual is mainly P3.** C learns a sizable positive P3 residual gain, while P4 converges near zero and slightly negative. The next clean structural ablation is therefore P3-only Prompt residual with P4 kept auxiliary-only.
5. **Do not rank runtime from these files.** A was tested on GPU 1 and B/C on GPU 4. Re-benchmark all three on one idle GPU with the same warm-up and repeats.